# Week 3 — Search and A\*: Using an LLM as an Engineering Assistant

**CS F407 Artificial Intelligence · Lab 3**

A warehouse robot must get from the loading area `S` to the delivery area `G`, moving Up/Down/Left/Right at cost 1 per move. This notebook works through the lab's tasks:

- **Task 0:** formulate the search problem.
- **Task 1:** design the agent.
- **Task 2:** have an LLM generate A\*.
- **Task 3:** test it systematically.
- **Task 4:** locate every A\* concept in the code.
- **Task 5:** compare A\* with BFS.
- **Task 6:** investigate the heuristic experimentally.
- **Task 7 and the Final Reflection:** evaluate the LLM's role.

The answers are inline and also collected in [`answers-to-questions/answers.md`](../answers-to-questions/answers.md). The prompts used are in [`prompts.md`](prompts.md). The code uses only the Python standard library.

In [1]:
import heapq
import math
import random
from collections import deque
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

WAREHOUSE_MAP = '''
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
'''

TRIVIAL_MAP = '''
#####
#SG##
#####
'''

NO_SOLUTION_MAP = '''
#######
#S....#
###.###
#...#G#
#######
'''

## Task 0 — Understand the Search Problem

| Component | Specification |
|---|---|
| **State $S$** | the robot's position $(r, c)$ (row, column) on a free cell of the grid: $S = \{(r,c) : \text{map}[r][c] \neq \#\}$. For this map that is 64 states. |
| **Actions $A$** | $\{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$, with displacements $(-1,0), (+1,0), (0,-1), (0,+1)$. In state $s$ the applicable actions $A(s)$ are those whose target cell is inside the grid and free. |
| **Transition $T$** | $T((r,c), a) = (r + \Delta r_a,\ c + \Delta c_a)$ for $a \in A(s)$. Deterministic: exactly one successor per applicable action. |
| **Initial state $s_0$** | the position of `S`: $(1, 1)$ |
| **Goal $G$** | $\{(7, 15)\}$, the position of `G`; goal test $s = (7, 15)$ |
| **Cost $c$** | $c(s, a, s') = 1$ for every move; path cost = number of moves |

**(a) What information is necessary to specify a state?**
Only the robot's position $(r, c)$. The map, the goal and the costs are fixed, so they are part of the *problem*, not of the state. Nothing else changes as the robot moves: no battery, orientation or carried load is modelled. Two different routes that arrive at the same cell are therefore the same state, which is what makes duplicate detection valid.

**(b) What makes an action invalid?**
An action is invalid in state $s$ if its target cell is an obstacle (`#`) or lies outside the grid. On this map the outer wall means the grid bounds are never actually reached, but the check should still be there for maps without a border.

**(c) Is this a deterministic search problem?**
Yes. The transition function gives exactly one successor for each (state, applicable action) pair. The environment is also fully observable (the map is known), static (the shelves do not move) and discrete. So the solution can be computed completely in advance, as a fixed action sequence, without needing to sense during execution.

**(d) What would constitute a solution?**
A sequence of actions $a_1, \dots, a_k$ such that each $a_i$ is applicable in the state reached so far and the final state is `G`. Equivalently, a sequence of adjacent free cells from $(1,1)$ to $(7,15)$. An **optimal** solution is one with minimum path cost, i.e. the fewest moves. If `G` is unreachable, the correct output is a report that no solution exists.

## Task 1 — Plan the Agent (designed before prompting)

1. **State representation:** a tuple `(row, col)`. Tuples are immutable and hashable, so they can be used as dictionary keys and set members.
2. **Warehouse representation:** a list of strings, `grid[row][col]`, parsed from the ASCII map. It is validated first: rectangular, exactly one `S` and one `G`, only the characters `#.SG`. The positions of `S` and `G` are extracted while parsing.
3. **Valid actions:** `ACTIONS = {"Up": (-1, 0), ...}`. `successors(state)` yields `(action, next_state)` for each action whose target is in bounds and not `#`.
4. **Goal recognition:** `state == goal`, tested when a state is **removed from the frontier for expansion**, not when it is generated. For A\* this ordering is what guarantees optimality.
5. **Frontier contents:** a binary heap of entries `(f, tie_break, g, state)`, where `f = g + h`. The cost-so-far `g` for each state is kept in a dictionary `g_cost`. A `closed` set records expanded states. When a cheaper path to a state is found, a new entry is pushed and the old one is skipped when it is popped ("lazy deletion").
6. **Path reconstruction:** a `parent` dictionary maps each state to `(previous_state, action)`. When the goal is expanded, follow the parents back to `S` and reverse.

**Reported on termination:** whether a solution was found, the path (as positions and as actions), the path length in **moves**, and the number of states **expanded** (popped and processed, not merely generated).

## Task 2 — Ask an LLM to Generate A\*

The prompt was the lab's example prompt with the Task 1 design details added. The full text is in [`prompts.md`](prompts.md), prompt 1. The code below is the core of what the LLM returned; the explanatory comments are trimmed, but the logic is unchanged.

In [2]:
def astar_first_attempt(grid_text):
    """A* search on a grid map (LLM first attempt)."""
    grid = [list(line) for line in grid_text.strip().split("\n")]
    rows, cols = len(grid), len(grid[0])
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "S":
                start = (r, c)
            elif grid[r][c] == "G":
                goal = (r, c)

    def h(state):                                  # Manhattan distance heuristic
        (x, y), (xg, yg) = state, goal
        return abs(x - xg) + abs(y - yg)

    open_heap = [(h(start), 0, start)]             # (f, g, state)
    came_from = {start: None}
    g_score = {start: 0}
    closed = set()
    expanded = 0

    while open_heap:
        f, g, current = heapq.heappop(open_heap)
        expanded += 1
        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return {"found": True, "path": path, "length": len(path), "expanded": expanded}
        closed.add(current)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nbr = (current[0] + dr, current[1] + dc)
            if not (0 <= nbr[0] < rows and 0 <= nbr[1] < cols) or grid[nbr[0]][nbr[1]] == "#":
                continue
            if nbr in closed:
                continue
            tentative_g = g + 1
            if tentative_g < g_score.get(nbr, float("inf")):
                g_score[nbr] = tentative_g
                came_from[nbr] = current
                heapq.heappush(open_heap, (tentative_g + h(nbr), tentative_g, nbr))

    return {"found": False, "path": None, "length": None, "expanded": expanded}


first = astar_first_attempt(WAREHOUSE_MAP)
print({k: v for k, v in first.items() if k != "path"})

{'found': True, 'length': 41, 'expanded': 64}


The first attempt runs and reports a path. The Task 3 tests are run against it *before* anything else is changed.

## Task 3 — Test the Generated Program

Every test needs an **independent** oracle for the expected answer, meaning something other than the code under test. Here the oracles are a hand-derivable answer (Tests 2 and 3), a plain BFS written separately, and the problem's own rules: each step must move to an adjacent free cell.

In [3]:
def parse(map_text):
    grid = map_text.strip("\n").split("\n")
    find = lambda ch: [(r, c) for r, row in enumerate(grid) for c, x in enumerate(row) if x == ch]
    return grid, find("S")[0], find("G")[0]

def bfs_distance(map_text):
    # Oracle: plain BFS shortest distance in moves, written independently of the A* code.
    grid, s, g = parse(map_text)
    dist, q = {s: 0}, deque([s])
    while q:
        r, c = q.popleft()
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            n = (r + dr, c + dc)
            if 0 <= n[0] < len(grid) and 0 <= n[1] < len(grid[0]) and grid[n[0]][n[1]] != "#" and n not in dist:
                dist[n] = dist[(r, c)] + 1
                q.append(n)
    return dist.get(g)

def path_is_valid(map_text, path):
    # Rules check: starts at S, ends at G, every step is to an adjacent free cell.
    grid, s, g = parse(map_text)
    steps_ok = all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 and grid[b[0]][b[1]] != "#" for a, b in zip(path, path[1:]))
    return path[0] == s and path[-1] == g and steps_ok

ALT_PATHS_MAP = '''
#######
#S...G#
#.###.#
#.....#
#######
'''
OPEN_ROOM_MAP = '''
#########
#S......#
#.......#
#.......#
#......G#
#########
'''

In [4]:
def run_tests(solver, label):
    print(f"=== {label} ===")
    results = {}
    cases = [
        ("Test 1: original warehouse", WAREHOUSE_MAP),
        ("Test 2: trivial (adjacent)", TRIVIAL_MAP),
        ("Test 3: no solution", NO_SOLUTION_MAP),
        ("Test 4a: two routes (4 vs 8 moves)", ALT_PATHS_MAP),
        ("Test 4b: open room (many equal paths)", OPEN_ROOM_MAP),
    ]
    for name, m in cases:
        res = solver(m)
        expected = bfs_distance(m)
        if expected is None:
            ok = res["found"] is False
            detail = f"found={res['found']} (expected no solution)"
        else:
            ok = res["found"] and path_is_valid(m, res["path"]) and res["length"] == expected
            detail = f"found={res['found']}, reported length={res['length']}, true shortest={expected}, expanded={res['expanded']}"
        results[name] = ok
        print(f"{'PASS' if ok else 'FAIL'}  {name:<40} {detail}")
    return results

first_results = run_tests(astar_first_attempt, "LLM first attempt")

=== LLM first attempt ===
FAIL  Test 1: original warehouse               found=True, reported length=41, true shortest=40, expanded=64
FAIL  Test 2: trivial (adjacent)               found=True, reported length=2, true shortest=1, expanded=2
PASS  Test 3: no solution                      found=False (expected no solution)
FAIL  Test 4a: two routes (4 vs 8 moves)       found=True, reported length=5, true shortest=4, expanded=5
FAIL  Test 4b: open room (many equal paths)    found=True, reported length=10, true shortest=9, expanded=28


### What the tests found in the first attempt

- **Test 2 fails, and it exposes an off-by-one error that affects every result.** On the trivial map the program reports a path "length" of **2**, but the solution is a single move. The code returns `len(path)`, where `path` is the list of *cells* including `S`, so every reported length is one more than the number of moves. Tests 1 and 4 fail for the same reason (41 reported for a 40-move path, 5 for 4, and 10 for 9). The paths themselves were correct and optimal: `path_is_valid` passes, and cells − 1 equals the BFS distance. Without a case whose answer is obvious (one move), "41" on the original map looks perfectly plausible.
- **Test 3 passes:** with `G` unreachable the frontier empties and the function returns `found: False`, so there is no infinite loop.
- **An inspection bug (see Task 4e):** the loop never checks whether the popped state is already in `closed`. When a state has been pushed twice (first by a worse path, then by a better one), the stale entry is popped later, **counted as expanded again**, and its neighbours are re-examined. None of the paths change, because the `closed` check on neighbours and the `g_score` comparison stop anything from being updated. But the "states expanded" measure, which Task 5 and Task 6 depend on, is inflated. The tests above do not show it, because with the Manhattan heuristic on these maps no state happens to be pushed twice. It appears in Task 6 with the inadmissible heuristic, as the check below demonstrates by comparing `expanded` with the number of *distinct* states expanded.

In [5]:
def count_distinct_first_attempt(grid_text, weight):
    # Re-run the first attempt's loop with a weighted heuristic, counting pops and distinct states.
    grid, start, goal = parse(grid_text)
    h = lambda s: weight * (abs(s[0] - goal[0]) + abs(s[1] - goal[1]))
    heap, g_score, closed, pops, distinct = [(h(start), 0, start)], {start: 0}, set(), 0, set()
    while heap:
        f, g, cur = heapq.heappop(heap)
        pops += 1
        distinct.add(cur)
        if cur == goal:
            break
        closed.add(cur)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            n = (cur[0] + dr, cur[1] + dc)
            if grid[n[0]][n[1]] == "#" or n in closed:
                continue
            if g + 1 < g_score.get(n, math.inf):
                g_score[n] = g + 1
                heapq.heappush(heap, (g + 1 + h(n), g + 1, n))
    return pops, len(distinct)

for w in [1, 2]:
    pops, distinct = count_distinct_first_attempt(WAREHOUSE_MAP, w)
    print(f"h = {w} x Manhattan: 'expanded' counter = {pops}, distinct states actually expanded = {distinct}")

h = 1 x Manhattan: 'expanded' counter = 64, distinct states actually expanded = 64
h = 2 x Manhattan: 'expanded' counter = 65, distinct states actually expanded = 64


### Corrected implementation

The prompt was updated with the failing test and the inspection finding ([`prompts.md`](prompts.md), prompt 2). The accepted version is below. It keeps the LLM's overall structure and data structures, which were correct, and makes these changes:
- **Path length:** reported as the number of **moves**, `len(path) - 1`, with the actions returned as well.
- **Stale entries:** a popped state that is already in `closed` is **skipped**, so `expanded` counts each state at most once.
- **Pluggable heuristic:** the heuristic is a parameter, which Task 6 needs.
- **Explicit f-value:** $f$ is computed and stored explicitly in each frontier entry.
- **Deterministic ties:** equal $f$ values are broken by a counter (first in, first out), not by comparing states.
- **Input validation**, and a shared `SearchResult` record so BFS and A\* report identical fields.

In [6]:
ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


class Warehouse:
    """The search problem P = (S, A, T, s0, G, c) for a grid map."""

    def __init__(self, map_text):
        grid = map_text.strip("\n").split("\n")
        if len({len(row) for row in grid}) != 1:
            raise ValueError("map rows must all have the same length")
        if set("".join(grid)) - set("#.SG"):
            raise ValueError("map may only contain # . S G")
        found = {ch: [(r, c) for r, row in enumerate(grid) for c, x in enumerate(row) if x == ch] for ch in "SG"}
        if len(found["S"]) != 1 or len(found["G"]) != 1:
            raise ValueError("map must contain exactly one S and one G")
        self.grid = grid
        self.n_rows, self.n_cols = len(grid), len(grid[0])
        self.initial = found["S"][0]                       # s0
        self.goal = found["G"][0]                          # G

    def is_goal(self, state):                              # goal test
        return state == self.goal

    def successors(self, state):                           # A(s) and T(s, a)
        r, c = state
        for action, (dr, dc) in ACTIONS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n_rows and 0 <= nc < self.n_cols and self.grid[nr][nc] != "#":
                yield action, (nr, nc), 1                  # cost c(s, a, s') = 1

    def render(self, path=()):
        mark = set(path) - {self.initial, self.goal}
        return "\n".join("".join("*" if (r, c) in mark else ch for c, ch in enumerate(row))
                         for r, row in enumerate(self.grid))


class SearchResult:
    def __init__(self, found, path, actions, expanded):
        self.found, self.path, self.actions, self.expanded = found, path, actions, expanded
        self.length = len(actions) if found else None       # number of moves

    def __repr__(self):
        return f"found={self.found}, length={self.length}, expanded={self.expanded}"


def reconstruct(parent, state):                            # path reconstruction
    path, actions = [state], []
    while parent[state] is not None:
        state, action = parent[state]
        path.append(state)
        actions.append(action)
    return path[::-1], actions[::-1]


# Heuristics h(n): estimated cost from state n to the goal
def h_manhattan(s, g): return abs(s[0] - g[0]) + abs(s[1] - g[1])
def h_zero(s, g):      return 0
def h_euclidean(s, g): return math.hypot(s[0] - g[0], s[1] - g[1])
def h_double(s, g):    return 2 * h_manhattan(s, g)


def astar(problem, h=h_manhattan):
    """A* graph search. Frontier: binary heap ordered by f = g + h."""
    start, goal = problem.initial, problem.goal
    g_cost = {start: 0}                                    # g(n)
    parent = {start: None}
    closed = set()                                         # visited (expanded) states
    counter = 0
    frontier = [(0 + h(start, goal), counter, start)]      # (f, tie-break, state)
    expanded = 0
    while frontier:
        f, _, state = heapq.heappop(frontier)              # select lowest f
        if state in closed:                                # stale entry: already expanded more cheaply
            continue
        closed.add(state)
        expanded += 1
        if problem.is_goal(state):                         # goal test on expansion
            return SearchResult(True, *reconstruct(parent, state), expanded)
        for action, nxt, cost in problem.successors(state):
            new_g = g_cost[state] + cost
            if nxt not in closed and new_g < g_cost.get(nxt, math.inf):
                g_cost[nxt] = new_g
                parent[nxt] = (state, action)
                counter += 1
                f_nxt = new_g + h(nxt, goal)               # f(n) = g(n) + h(n)
                heapq.heappush(frontier, (f_nxt, counter, nxt))
    return SearchResult(False, None, None, expanded)


def bfs(problem):
    """Breadth-first graph search. Frontier: FIFO queue. Goal test on expansion for a like-for-like count."""
    start = problem.initial
    parent = {start: None}                                 # reached set + parents
    frontier = deque([start])
    expanded = 0
    while frontier:
        state = frontier.popleft()
        expanded += 1
        if problem.is_goal(state):
            return SearchResult(True, *reconstruct(parent, state), expanded)
        for action, nxt, _ in problem.successors(state):
            if nxt not in parent:
                parent[nxt] = (state, action)
                frontier.append(nxt)
    return SearchResult(False, None, None, expanded)

In [7]:
def as_dict(solver):
    def run(m):
        r = solver(Warehouse(m))
        return {"found": r.found, "path": r.path, "length": r.length, "expanded": r.expanded}
    return run

final_results = run_tests(as_dict(astar), "Corrected A*")
assert all(final_results.values())

# Malformed input is rejected clearly
for bad in [WAREHOUSE_MAP.replace("S", "."), WAREHOUSE_MAP.replace("#.###.#.#######.#", "#.###.#")]:
    try:
        Warehouse(bad)
    except ValueError as e:
        print("ValueError:", e)

=== Corrected A* ===
PASS  Test 1: original warehouse               found=True, reported length=40, true shortest=40, expanded=64
PASS  Test 2: trivial (adjacent)               found=True, reported length=1, true shortest=1, expanded=2
PASS  Test 3: no solution                      found=False (expected no solution)
PASS  Test 4a: two routes (4 vs 8 moves)       found=True, reported length=4, true shortest=4, expanded=5
PASS  Test 4b: open room (many equal paths)    found=True, reported length=9, true shortest=9, expanded=28
ValueError: map must contain exactly one S and one G
ValueError: map rows must all have the same length


In [8]:
wh = Warehouse(WAREHOUSE_MAP)
res = astar(wh)
print(f"Test 1 (original warehouse): solution found = {res.found}, path length = {res.length} moves, states expanded = {res.expanded}\n")
print(wh.render(res.path), "\n")
print("Path (cells):", " -> ".join(f"({r},{c})" for r, c in res.path), "\n")
print("Actions:", ", ".join(res.actions))
(OUT_DIR / "astar_path.txt").write_text(
    f"A* (Manhattan) on the lab warehouse: {res.length} moves, {res.expanded} states expanded\n\n"
    + wh.render(res.path) + "\n\nActions: " + ", ".join(res.actions) + "\n")

for name, m in [("Test 2", TRIVIAL_MAP), ("Test 3", NO_SOLUTION_MAP), ("Test 4a", ALT_PATHS_MAP), ("Test 4b", OPEN_ROOM_MAP)]:
    w = Warehouse(m); r = astar(w)
    print(f"\n{name}: {r}   actions={r.actions}")
    if r.found:
        print(w.render(r.path))

Test 1 (original warehouse): solution found = True, path length = 40 moves, states expanded = 64

#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
################# 

Path (cells): (1,1) -> (1,2) -> (1,3) -> (1,4) -> (1,5) -> (2,5) -> (3,5) -> (4,5) -> (5,5) -> (5,6) -> (5,7) -> (5,8) -> (5,9) -> (5,10) -> (5,11) -> (5,12) -> (5,13) -> (4,13) -> (3,13) -> (3,12) -> (3,11) -> (3,10) -> (3,9) -> (3,8) -> (3,7) -> (2,7) -> (1,7) -> (1,8) -> (1,9) -> (1,10) -> (1,11) -> (1,12) -> (1,13) -> (1,14) -> (1,15) -> (2,15) -> (3,15) -> (4,15) -> (5,15) -> (6,15) -> (7,15) 

Actions: Right, Right, Right, Right, Down, Down, Down, Down, Right, Right, Right, Right, Right, Right, Right, Right, Up, Up, Left, Left, Left, Left, Left, Left, Up, Up, Right, Right, Right, Right, Right, Right, Right, Right, Down, Down, Down, Down, Down, Down

Test 2: found=True, length=1, expanded=2   actions=['Right']
#####
#SG##
#

**Test results (corrected A\*):**

| Test | Found? | Length (moves) | Expected | States expanded | Result |
|---|---|---|---|---|---|
| 1. Original warehouse | yes | 40 | 40 (BFS) | 64 | pass |
| 2. Trivial | yes | 1 (`Right`) | 1 | 2 | pass |
| 3. No solution | no, reported as failure | – | no solution | 9 (every reachable cell) | pass (terminates) |
| 4a. Two routes | yes | 4 (top route) | 4, not the 8-move bottom route | 5 | pass |
| 4b. Open room | yes | 9 | 9 = Manhattan distance | 28 | pass |

Test 4b has a side lesson about **tie-breaking**. In an empty room Manhattan distance is *exact*, so every one of the 28 cells has $f = g + h = 9 = C^*$, and there are $\binom{9}{3} = 84$ shortest paths. With ties broken first-in-first-out, A\* expands **all 28** equally-good cells before it reaches `G`. Breaking ties towards larger $g$ (deeper nodes) goes almost straight to the goal: it expands 10 states instead of 28 (checked separately). The path is optimal either way; only the effort differs.

### Think About It — working output ≠ validated algorithm

The first attempt produced plausible output on the real map: a path that looked right, a length of 41, 64 states expanded. **Every one of those numbers was reported without a check, and one was wrong.** The error only became visible on a case with a known answer (a one-move map). The inflated expansion count only became visible by comparing the counter against an independent quantity (distinct states). Tests with known expected behaviour turn "it printed something reasonable" into evidence.

## Task 4 — Inspect the A\* Algorithm

Where each concept appears in the final code:

| Concept | Where it appears |
|---|---|
| **State** | a `(row, col)` tuple; `Warehouse.initial`, the `state` / `nxt` variables in `astar` |
| **Action** | the `ACTIONS` dictionary (name → `(d_row, d_col)`); `action` in `Warehouse.successors` |
| **Transition** | `Warehouse.successors`: computes `(nr, nc) = (r + dr, c + dc)` and keeps only in-bounds, non-`#` cells |
| **Goal test** | `Warehouse.is_goal`, called in `astar` right after a state is popped for expansion |
| **$g(n)$** | the `g_cost` dictionary; updated as `new_g = g_cost[state] + cost` |
| **$h(n)$** | the heuristic function passed as `h` (default `h_manhattan`); called as `h(nxt, goal)` when a successor is pushed |
| **$f(n)$** | `f_nxt = new_g + h(nxt, goal)`, stored as the first element of each heap entry |
| **Frontier** | `frontier`: a list managed as a binary min-heap by `heapq.heappush` / `heapq.heappop` |
| **Visited states** | `closed`, the set of expanded states; stale heap entries for closed states are skipped |
| **Path reconstruction** | the `parent` dictionary (state → `(previous state, action)`) and `reconstruct()` |

**(a) What data structure is used for the A\* frontier?**
A **binary heap** (priority queue), implemented as a Python list with the `heapq` module. Each entry is a tuple `(f, counter, state)`. Tuples compare element by element, so the heap orders by `f`, then by insertion order. Push and pop are $O(\log n)$.

**(b) How does the program select the next state to expand?**
`heapq.heappop(frontier)` removes the entry with the **smallest $f = g + h$**. Ties go to the earliest-inserted entry. If the popped state is already in `closed`, it is a stale duplicate and is discarded, and the next entry is popped.

**(c) Where is the heuristic calculated?**
When a successor is generated and pushed: `h(nxt, goal)` inside the successor loop, and once for the start state when the frontier is initialised. Each value is computed as the entry is created, not recomputed later. The LLM's first attempt did the same, with an inner function `h(state)`.

**(d) Does the program explicitly calculate $f(n) = g(n) + h(n)$?**
Yes: `f_nxt = new_g + h(nxt, goal)`, stored in the heap entry. In the first attempt it was calculated inline, as `tentative_g + h(nbr)` inside the `heappush` call, so it was computed but never named.

**(e) How does the program prevent unnecessary repeated exploration?**
- **The `closed` set.** Once a state is expanded it is never expanded again, and successors already in `closed` are not pushed.
- **The `g_cost` check.** A successor is pushed only if the new path to it is **strictly cheaper** than any path found before.
- **Skipping stale entries** when popped: `if state in closed: continue`.

The first attempt had the first two but **not the third**. It could pop and re-expand a stale duplicate, and it counted that as an extra expansion (65 "expanded" for 64 distinct states with $h = 2\times$ Manhattan).

## Task 5 — Compare A\* with Blind Search

BFS was produced from the same design (prompt 3 in [`prompts.md`](prompts.md)), with the warehouse unchanged. Both use the same `Warehouse` problem object and the same definition of "expanded": a state removed from the frontier and processed, goal test included.

In [9]:
res_bfs, res_astar = bfs(wh), astar(wh)
print(f"{'Measure':<18}{'BFS':>8}{'A*':>8}")
print(f"{'Solution found':<18}{str(res_bfs.found):>8}{str(res_astar.found):>8}")
print(f"{'Path length':<18}{res_bfs.length:>8}{res_astar.length:>8}")
print(f"{'States expanded':<18}{res_bfs.expanded:>8}{res_astar.expanded:>8}")
print(f"\nfree cells in the warehouse: {sum(ch != '#' for row in wh.grid for ch in row)}")
print("same path?", res_bfs.path == res_astar.path)

Measure                BFS      A*
Solution found        True    True
Path length             40      40
States expanded         64      64

free cells in the warehouse: 64
same path? True


The two algorithms expanded exactly the same number of states on this map: **every free cell**. That is a real result, not a bug, and the analysis below shows why.

In [10]:
def exact_costs_to_goal(problem):
    # h*(n): true cost from every state to the goal (BFS backwards from G; moves are reversible).
    d, q = {problem.goal: 0}, deque([problem.goal])
    while q:
        s = q.popleft()
        for _, n, _ in problem.successors(s):
            if n not in d:
                d[n] = d[s] + 1
                q.append(n)
    return d

h_star = exact_costs_to_goal(wh)                  # true cost from each state to G
dist_from_s = {wh.initial: 0}                      # g*(n): true cost from S to each state
q = deque([wh.initial])
while q:
    s = q.popleft()
    for _, n, _ in wh.successors(s):
        if n not in dist_from_s:
            dist_from_s[n] = dist_from_s[s] + 1
            q.append(n)

C_star = h_star[wh.initial]
must_expand = [s for s in h_star if dist_from_s[s] + h_manhattan(s, wh.goal) < C_star]
on_optimal = [s for s in h_star if dist_from_s[s] + h_star[s] == C_star]
print(f"optimal cost C* = {C_star}; Manhattan h(S) = {h_manhattan(wh.initial, wh.goal)} (the maze doubles the straight-line estimate)")
print(f"states with g*(n) + h(n) < C*  (A* with Manhattan MUST expand these): {len(must_expand)} of {len(h_star)}")
print(f"states lying on an optimal path: {len(on_optimal)}")
print(f"states with g*(n) > C* (farther from S than the goal is; BFS never needs these): {sum(dist_from_s[s] > C_star for s in h_star)}")
print(f"max distance from S of any free cell: {max(dist_from_s.values())}")

optimal cost C* = 40; Manhattan h(S) = 20 (the maze doubles the straight-line estimate)
states with g*(n) + h(n) < C*  (A* with Manhattan MUST expand these): 49 of 64
states lying on an optimal path: 41
states with g*(n) > C* (farther from S than the goal is; BFS never needs these): 0
max distance from S of any free cell: 40


**(a) Did both algorithms find a solution?** Yes.

**(b) Did they find paths of the same length?** Yes, 40 moves. In fact it is the same path. Both are optimal here: BFS because every move costs 1, and A\* because Manhattan distance is admissible.

**(c) Which algorithm expanded fewer states?** Neither. **Both expanded all 64 free cells.**

**(d) Why might A\* expand fewer states, and why doesn't it here?**
In general A\* expands fewer states because it uses **information about where the goal is**. BFS expands in order of $g$ alone, spreading equally in all directions. A\* expands in order of $g + h$, so states that are "far from the goal" according to $h$ are postponed, and many are never expanded before the goal is reached. A\* with an admissible $h$ must expand every state with $f(n) = g^*(n) + h(n) < C^*$, but only those (plus some ties). So the better $h$ approximates the true remaining cost, the fewer states satisfy that inequality.

On *this* warehouse the heuristic is badly misleading:
- The map is a maze with essentially one route, so the optimal path is 40 moves while the Manhattan estimate from `S` is only 20.
- The route has to head away from `G` for long stretches.
- Because the true cost is twice the estimate, **49 of the 64 states have $g^* + h < C^* = 40$**, so A\* is *obliged* to expand them. The remaining 15 all tie at exactly $f = 40$, and first-in-first-out tie-breaking expands them before the goal.
- BFS, for its part, expands every state closer to `S` than the goal is. The goal happens to be the farthest cell from `S` (distance 40), and only 41 cells lie on the single optimal path.

So both searches end up covering the whole warehouse. A\*'s advantage depends on how informative $h$ is on the specific map. The larger random warehouses in Task 6 show the usual gap.

### Think About It — what information does the algorithm use to decide where to search next?

- **BFS** uses only the order in which states were discovered, which amounts to $g$, the depth. It uses no information about the goal apart from the goal test.
- **A\*** uses $g(n)$, the cost already paid, plus $h(n)$, an **estimate of the cost still to go**, which uses the goal's position.

That extra information is only as good as the estimate. When the estimate is close to the truth (open floor), A\* heads almost straight for the goal. When the walls make it wildly optimistic (this maze), the ordering it induces is barely better than BFS's. A\* is "more intelligent" only to the extent that $h$ carries real information about the problem.

## Task 6 — Investigate the Heuristic

### Why Manhattan distance suits this warehouse (the LLM's explanation, checked)

Asked why Manhattan distance is appropriate when the robot can only move horizontally and vertically, the LLM's explanation (prompt 4 in [`prompts.md`](prompts.md)) came down to three points. Each was checked against the experiments rather than taken on trust:

1. **It is exactly the cost with no obstacles.** Each move changes the row *or* the column by 1, so reaching the goal needs at least $|r - r_G| + |c - c_G|$ moves. This is confirmed by Test 4b, where the path length equals the Manhattan distance.
2. **It is admissible.** Obstacles can only make the true cost *larger*, never smaller, so $h(n) \le h^*(n)$ everywhere, and A\* returns optimal paths. This is checked below for every state.
3. **It is consistent.** One move changes $h$ by exactly 1 and costs 1, so $h(n) \le c(n, n') + h(n')$, and every state is expanded at most once with its optimal $g$. Also checked below.

It also dominates Euclidean distance ($|dx| + |dy| \ge \sqrt{dx^2 + dy^2}$): both are admissible, but Manhattan is never smaller, so it is at least as informative.

### Experiments on the lab warehouse

In [11]:
HEURISTICS = {
    "h = 0 (uniform-cost)": h_zero,
    "Manhattan": h_manhattan,
    "Euclidean": h_euclidean,
    "2 x Manhattan": h_double,
}

def admissibility_report(problem, h):
    hs = exact_costs_to_goal(problem)
    over = [s for s in hs if h(s, problem.goal) > hs[s] + 1e-9]
    inconsistent = sum(1 for s in hs for _, n, c in problem.successors(s)
                       if h(s, problem.goal) > c + h(n, problem.goal) + 1e-9)
    return len(over), len(hs), inconsistent

print(f"{'heuristic':<22}{'found':>7}{'length':>8}{'expanded':>10}{'  h > h* at':>13}{'  inconsistent edges':>22}")
for name, h in HEURISTICS.items():
    r = astar(wh, h)
    over, n, inc = admissibility_report(wh, h)
    print(f"{name:<22}{str(r.found):>7}{r.length:>8}{r.expanded:>10}{f'{over}/{n} states':>13}{inc:>22}")
print(f"{'BFS (reference)':<22}{str(res_bfs.found):>7}{res_bfs.length:>8}{res_bfs.expanded:>10}")

heuristic               found  length  expanded    h > h* at    inconsistent edges
h = 0 (uniform-cost)     True      40        64  0/64 states                     0
Manhattan                True      40        64  0/64 states                     0
Euclidean                True      40        64  0/64 states                     0
2 x Manhattan            True      40        64 18/64 states                    64
BFS (reference)          True      40        64


On the lab warehouse all four heuristics find the same 40-move path with 64 expansions. The $2\times$ heuristic *is* inadmissible here: it overestimates at 18 of the 64 states, and it violates consistency on every edge. But the maze has only one sensible route, so there is no shorter alternative for the overestimate to make A\* skip. **A single map cannot answer the question.** The same experiment was therefore repeated on 50 solvable random warehouses of 21 × 41 cells, with shelving aisles and scattered clutter. There, alternative routes and open floor exist.

In [12]:
def random_warehouse(n_rows, n_cols, seed, gap=0.25, clutter=0.08):
    rng = random.Random(seed)
    g = [["#"] * n_cols] + [["#"] + ["."] * (n_cols - 2) + ["#"] for _ in range(n_rows - 2)] + [["#"] * n_cols]
    for r in range(2, n_rows - 2, 2):                  # shelving rows with random gaps
        for c in range(1, n_cols - 1):
            if rng.random() > gap:
                g[r][c] = "#"
    for r in range(1, n_rows - 1):                     # scattered clutter
        for c in range(1, n_cols - 1):
            if g[r][c] == "." and rng.random() < clutter:
                g[r][c] = "#"
    g[1][1], g[n_rows - 2][n_cols - 2] = "S", "G"
    return "\n".join("".join(row) for row in g)

maps = []
seed = 0
while len(maps) < 50:
    m = random_warehouse(21, 41, seed)
    if bfs(Warehouse(m)).found:
        maps.append(m)
    seed += 1

summary = {name: {"expanded": 0, "suboptimal": 0, "excess": 0, "found": 0} for name in ["BFS", *HEURISTICS]}
for m in maps:
    w = Warehouse(m)
    opt = bfs(w)
    for name, solver in [("BFS", bfs), *[(k, (lambda h: lambda p: astar(p, h))(h)) for k, h in HEURISTICS.items()]]:
        r = solver(w)
        s = summary[name]
        s["found"] += r.found
        s["expanded"] += r.expanded
        s["suboptimal"] += r.length > opt.length
        s["excess"] += r.length - opt.length

print(f"50 random 21x41 warehouses (optimal length averages {sum(bfs(Warehouse(m)).length for m in maps) / 50:.1f} moves)\n")
print(f"{'algorithm / heuristic':<24}{'found':>7}{'mean expanded':>15}{'suboptimal':>12}{'mean extra moves':>18}")
for name, s in summary.items():
    print(f"{name:<24}{s['found']:>5}/50{s['expanded'] / 50:>15.1f}{s['suboptimal']:>9}/50{s['excess'] / 50:>18.2f}")

with open(OUT_DIR / "heuristic_experiment.txt", "w") as f:
    for name, s in summary.items():
        f.write(f"{name}: found {s['found']}/50, mean expanded {s['expanded'] / 50:.1f}, "
                f"suboptimal {s['suboptimal']}/50, mean extra moves {s['excess'] / 50:.2f}\n")

50 random 21x41 warehouses (optimal length averages 60.5 moves)

algorithm / heuristic     found  mean expanded  suboptimal  mean extra moves
BFS                        50/50          404.8        0/50              0.00
h = 0 (uniform-cost)       50/50          404.8        0/50              0.00
Manhattan                  50/50          225.6        0/50              0.00
Euclidean                  50/50          284.7        0/50              0.00
2 x Manhattan              50/50          100.9       17/50              1.76


In [13]:
# One concrete case where the inflated heuristic returns a longer path
for m in maps:
    w = Warehouse(m)
    good, bad = astar(w, h_manhattan), astar(w, h_double)
    if bad.length - good.length >= 6:
        break
print(f"Manhattan: {good}\n{w.render(good.path)}\n")
print(f"2 x Manhattan: {bad}\n{w.render(bad.path)}")

Manhattan: found=True, length=56, expanded=210
#########################################
#S.#.....................#..............#
#*###.####..##.#...####.#####.#.######.##
#*************.......#........#.........#
###.######.##*########..######.###..#####
#..........#.******...........#.....#...#
##..##############*#..####.###..#########
#.....#...........*#....................#
##..##############*########....####.###.#
#....#..........#.***...................#
#..###.####.###.####*#######.########.###
#...................*...................#
####...####.#####..#*####.####.##########
#.#...#.............**............#....##
###.######.###.#.####*##.########.#.#####
#....................**...............#.#
#####.##.#.#####.##..#*####.###..##.#####
#.....#.........##....******************#
#.#.###..###########..##.#..###..######*#
#................#...........#....#....G#
#########################################



2 x Manhattan: found=True, length=62, expanded=88
#########################################
#S.#.....................#..............#
#*###.####..##.#...####.#####.#.######.##
#***.................#........#.........#
###*######.##.########..######.###..#####
#..*.......#..................#.....#...#
##.*##############.#..####.###..#########
#..*..#............#....................#
##.*##############.########....####.###.#
#.**.#..........#.......................#
#.*###.####.###.####.#######.########.###
#.**********............................#
####...####*#####..#.####.####.##########
#.#...#....****...................#....##
###.######.###*#.####.##.########.#.#####
#.............******..................#.#
#####.##.#.#####.##*.#.####.###..##.#####
#.....#.........##.**......*************#
#.#.###..###########*.##.#.*###..######*#
#................#..********.#....#....G#
#########################################


### Results and interpretation

**1. $h = 0$.** A\* becomes **uniform-cost search**. With unit costs this is BFS ordered by $g$. It always finds an optimal path, but it expands the most states, the same as BFS: every state with $g < C^*$. With no information about the goal, it searches "in circles" around the start.

**2. Euclidean distance.** Still **admissible**: a straight line is never longer than a 4-connected path, so it never overestimates, and every path found is optimal. But it is **smaller** than Manhattan wherever both $dx$ and $dy$ are non-zero, so it is less informative. More states satisfy $g + h < C^*$, and A\* expands more of them. On the random warehouses it falls between $h = 0$ and Manhattan.

**3. $2 \times$ Manhattan.** **Inadmissible**: it overestimates, and the check found $h > h^*$ at 18 of the 64 states of the lab warehouse. It expands **by far the fewest states**, about 101 on average versus 226 for Manhattan, because it pushes A\* strongly towards the goal, like greedy best-first search. The price is **optimality**: on 17 of the 50 random warehouses it returned a path longer than the shortest (1.8 extra moves on average, and 6 extra in the example above). A\* may now reach `G` along a route that *looks* direct and accept it before finishing the cheaper detour, whose $f$ was overestimated. This is weighted A\*: the path found is guaranteed to be at most $2\times$ optimal, and it is found faster.

| Heuristic | Admissible? | Optimal paths? | States expanded |
|---|---|---|---|
| $h = 0$ | yes (trivially) | always | most (same as BFS) |
| Euclidean | yes | always | fewer than $h = 0$ |
| Manhattan | yes, and consistent | always | fewer than Euclidean |
| $2\times$ Manhattan | **no** | often **not** | fewest |

### Think About It — too optimistic vs too aggressive

The experiments give the answer directly:
- **Too optimistic** (underestimates: $h = 0$, Euclidean) keeps A\* **correct**, returning optimal paths, but makes it **slow**. The weaker the estimate, the more A\* degrades towards blind uniform-cost search.
- **Too aggressive** (overestimates: $2\times$ Manhattan) makes A\* **fast but no longer optimal**. It trusts the estimate so much that it stops exploring alternatives that would have been cheaper.

Admissibility, $h \le h^*$, is exactly the boundary between the two. The ideal is the largest heuristic that still never overestimates: $h = h^*$ itself, which would expand only the states on an optimal path. Among admissible heuristics, larger is better, which is why Manhattan beats Euclidean here.

## Task 7 — Evaluate the LLM-Generated Agent

**1. What parts of the generated code were correct immediately?**
The core A\* algorithm:
- the heap-based frontier ordered by $f = g + h$;
- the Manhattan heuristic;
- the `g_score` dictionary with the "only push if strictly cheaper" rule;
- the `closed` set for successors;
- the goal test at expansion time;
- `came_from` path reconstruction;
- termination with failure when the frontier empties (Test 3).

Every path it returned was valid and optimal.

**2. Did you find any bugs or design problems?**
Yes, two bugs and some design problems:
- **Bug 1: off-by-one path length.** It reported `len(path)`, the number of cells, instead of the number of moves, so every length was 1 too high.
- **Bug 2: stale heap entries not skipped when popped.** A duplicate state could be re-expanded and counted again, inflating "states expanded".
- **Design problems:**
  - the heuristic was hard-wired, so it could not be varied for Task 6;
  - there was no input validation: a map without `S` raises `UnboundLocalError`;
  - the heuristic's variable names `(x, y)` actually held `(row, col)`. That is harmless for Manhattan distance, which is symmetric, but misleading.

**3. How did you discover those problems?**
- **Bug 1** came from **Test 2**. A one-move map reported length 2, which is obviously wrong once the answer is known in advance.
- **Bug 2** came from **reading the code for Task 4(e)** ("how does it prevent repeated exploration?"), and was then confirmed by **comparing the expansion counter with the number of distinct states** under $2\times$ Manhattan (65 vs 64).
- **The design problems** surfaced when trying to run the Task 6 experiments, and when feeding in a malformed map.

**4. Did the LLM use terminology or data structures that you did not understand?**
Two things needed working through:
- **`heapq`** behaves as a *min-heap on tuples*, so the tuple order `(f, g, state)` silently decides both priority and tie-breaking. Comparing `state` tuples on ties is legal, but arbitrary.
- **"Lazy deletion":** `heapq` has no decrease-key operation, so a cheaper path is handled by pushing a *new* entry and leaving the old one in the heap. That old entry is exactly what Bug 2 failed to skip.

Terms like "open set" and "closed set" match the lecture's frontier and explored set.

**5. Did you modify the LLM-generated code?**
Yes. The changes:
- the length is now reported in moves, and the actions are returned;
- stale entries are skipped;
- the heuristic is a parameter, with $f$ named explicitly;
- ties are broken deterministically by a counter;
- the problem is wrapped in a `Warehouse` class with validation, so it matches the $(S, A, T, s_0, G, c)$ formulation;
- a shared `SearchResult` record lets BFS and A\* be compared on identical measures.

The algorithmic core was kept.

**6. Which tests were most useful?**
- **Test 2 (trivial case)** was the most useful: it was the only one that exposed the off-by-one error, because it is the only case whose answer is obvious.
- **Independent oracles** came next: a separately written BFS for the true shortest length, the path-validity check, and the distinct-state count. They turn "looks plausible" into "is correct".
- **Test 4** (alternative paths, open room) checked optimality where a wrong algorithm would visibly differ.
- **Test 3** confirmed termination.
- The **random-map experiments** were essential for Task 6, where the lab map on its own was misleading.

**7. Could you have trusted the program without testing it?**
No. It ran first time, and every number it printed on the lab map looked reasonable, yet the reported length was wrong. Worse, the "states expanded" metric would have been subtly wrong in exactly the experiments (Tasks 5–6) whose conclusions depend on it. On the lab map alone, even a *correct* program gives the surprising "A\* = BFS" result. Without the analysis and extra maps, it would have been easy to wrongly blame the code, or to wrongly conclude that heuristics do not help.

**8. What did you understand about A\* that you did not understand before implementing it?**
- **A\* is not automatically faster than BFS.** Its advantage depends entirely on how informative $h$ is on the specific map. On a maze where the true cost is twice the estimate, it expands every state, just as BFS does.
- **Admissibility has a precise, measurable meaning:** every state with $g + h < C^*$ *must* be expanded. That explains both why a weak $h$ is slow and why an inflated $h$ can skip the optimal route.
- **The goal test must happen at expansion, not generation.** Testing at generation could accept a path before a cheaper one has been found.
- **Implementation details decide what "states expanded" means,** for example the lazy-deletion skip. The metric has to be defined precisely before two algorithms can be compared.

### What I designed, what the LLM suggested, what I accepted, changed and tested

| | |
|---|---|
| **Designed myself (before prompting)** | the search-problem formulation (Task 0); the state, map and frontier representation; the goal test on expansion; the parent-pointer reconstruction; the reporting requirements (Task 1); the test plan and oracles (Task 3); the heuristic experiments on random maps; the admissibility and consistency checks (Task 6) |
| **Suggested by the LLM** | the complete first A\* implementation, i.e. the `heapq` frontier with `(f, g, state)` tuples, `g_score`, `came_from`, the `closed` set and the Manhattan function; the BFS variant; the explanation of why Manhattan suits 4-connected movement |
| **Accepted** | the algorithmic core of A\* and BFS, and the three-point explanation of Manhattan, after each point was checked experimentally |
| **Changed** | length in moves; skipping stale entries; parameterised heuristic with explicit $f$; counter tie-breaking; `Warehouse` class with validation; shared `SearchResult` |
| **Tested** | Tests 1–4 against a separate BFS oracle and a path-validity check; malformed input; distinct vs counted expansions; admissibility and consistency of each heuristic over all states; BFS vs A\* and four heuristics on the lab map and on 50 random warehouses |

## Final Reflection

**1. Why is it important to formulate the search problem before writing the search algorithm?**
The formulation decides what the algorithm is *allowed* to assume. The state being just `(row, col)` is what makes duplicate detection correct. The unit costs are what make BFS optimal, and they are also what makes Manhattan distance admissible. Placing the goal test on the state tells you where it goes in the loop. Without the formulation, there is nothing to check generated code against. The two bugs found here were only recognisable as bugs because the specification said "length = number of moves" and "expanded = states removed from the frontier and processed". An LLM will happily produce code for an under-specified problem, and it will fill in the gaps with its own guesses.

**2. In what sense is A\* an "informed" search algorithm?**
It uses problem-specific knowledge beyond the problem definition: the heuristic $h(n)$, an estimate of the remaining cost to the goal that uses the goal's position. Blind searches (BFS, DFS, uniform-cost) choose the next state using only the states already generated and their depth or cost. A\* ranks states by $f = g + h$, the estimated cost of the *best solution through* $n$, so it spends its effort on states that look likely to lead to a cheap solution. With an admissible $h$ it does this without giving up optimality.

**3. Why does the choice of heuristic matter?**
The heuristic determines both **efficiency** and **correctness**:
- **Efficiency:** A\* must expand every state with $g + h < C^*$, so a more accurate admissible heuristic means fewer expansions. On the random warehouses the mean expansions were 226 for Manhattan, 285 for Euclidean and 405 for $h = 0$.
- **Correctness:** an overestimating heuristic can make A\* return a suboptimal path. $2\times$ Manhattan did so on a third of the random warehouses, though it expanded the fewest states (101).
- **Fit to the problem:** Manhattan is right for 4-connected unit-cost moves; for 8-connected or diagonal moves it would overestimate. On a maze like the lab map, even a good heuristic buys nothing.

The heuristic is where domain knowledge enters the algorithm. Choosing it means trading speed against guarantees, and that choice has to be justified for the problem at hand.

**4. What did the LLM contribute to the engineering process?**
- **A fast, largely correct first implementation** of a standard algorithm: the heap frontier, $g$ bookkeeping and reconstruction, in idiomatic Python.
- **Quick variants** (BFS) on request.
- **A clear explanation** of why Manhattan distance fits the movement model.

That saved most of the typing and let the time go into specification, testing and experiments. What it did *not* contribute was any assurance of correctness. It shipped an off-by-one in the headline metric and a subtle counting bug, and it could not have known the map-specific result that A\* = BFS here. Its explanations were useful as hypotheses to test, not as conclusions.

**5. What could go wrong if an engineer simply accepted LLM-generated code without testing it?**
Here specifically:
- every reported path length would have been wrong by one;
- the expansion counts behind the BFS-vs-A\* and heuristic comparisons would have been inflated under inconsistent heuristics;
- the conclusions drawn from them could have been wrong.

More generally:
- code that runs and prints plausible output can still violate the specification;
- edge cases (no path, malformed input) can crash or hang;
- subtle issues such as an inadmissible heuristic or a goal test at generation silently return suboptimal answers.

In a real warehouse robot these would mean wrong routes, wrong performance estimates, or failures on the first unusual layout. And the engineer would be unable to explain or defend the system's behaviour, because nobody ever checked it. The LLM-generated program is a hypothesis. Testing is what turns it into an engineering result.